In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [2]:
# Загрузка данных
df = pd.read_csv("../data/dataset_train.csv")

In [3]:
LE = LabelEncoder()
df['gender'] = LE.fit_transform(df['gender'])
df['product'] = LE.fit_transform(df['product'])

In [4]:
df['DateTime'] = pd.to_datetime(df['DateTime'])
df['DateTime'] = df['DateTime'].dt.hour

In [5]:
# Удалим session_id (не несёт полезной информации)
df = df.drop(columns=["session_id"])

# Заполним пропуски
df = df.fillna(df.median(numeric_only=True))

# Целевая переменная
target = "is_click"

X = df.drop(columns=[target])
y = df[target].values

# Масштабируем признаки
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [6]:
class ClickDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
train_dataset = ClickDataset(X_train, y_train)
val_dataset = ClickDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

In [8]:
class ClickModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [9]:
input_dim = X_train.shape[1]
model = ClickModel(input_dim)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()

            preds = torch.sigmoid(outputs)
            predicted = (preds > 0.5).float()
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    accuracy = correct / total
    return total_loss / len(loader), accuracy

In [12]:
epochs = 30

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f}")
    print("-" * 40)

Epoch 1/30
Train Loss: 0.2616
Val Loss: 0.2464 | Val Accuracy: 0.9323
----------------------------------------
Epoch 2/30
Train Loss: 0.2479
Val Loss: 0.2460 | Val Accuracy: 0.9323
----------------------------------------
Epoch 3/30
Train Loss: 0.2469
Val Loss: 0.2463 | Val Accuracy: 0.9323
----------------------------------------
Epoch 4/30
Train Loss: 0.2465
Val Loss: 0.2460 | Val Accuracy: 0.9323
----------------------------------------
Epoch 5/30
Train Loss: 0.2463
Val Loss: 0.2458 | Val Accuracy: 0.9323
----------------------------------------
Epoch 6/30
Train Loss: 0.2460
Val Loss: 0.2458 | Val Accuracy: 0.9323
----------------------------------------
Epoch 7/30
Train Loss: 0.2459
Val Loss: 0.2457 | Val Accuracy: 0.9323
----------------------------------------
Epoch 8/30
Train Loss: 0.2459
Val Loss: 0.2453 | Val Accuracy: 0.9323
----------------------------------------
Epoch 9/30
Train Loss: 0.2457
Val Loss: 0.2452 | Val Accuracy: 0.9323
----------------------------------------
E

In [13]:
torch.save(model.state_dict(), "click_model.pth")